# Hello Gyroid Sphere — Spectral Analysis

Visualize the frequency content of VoxelCAD signed distance fields.
This notebook creates a gyroid-sphere model and examines its power spectrum
to understand the relationship between geometry features, staircase artifacts,
and the Butterworth low-pass filter used in the EDT export pipeline.

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
logging.basicConfig(level=logging.INFO,
                    format='%(levelname)s %(name)s: %(message)s')

from voxelcad import Sphere, GyroidCube, ENV
from voxelcad.utils.spectral import (
    radial_power_spectrum, estimate_bandwidth, safe_stride,
    recommend_cutoff, apply_filter, FILTER_ZOO,
    plot_radial_spectrum, plot_filter_effect,
    plot_spectrum_slices, plot_cumulative_energy,
    plot_filter_transfer_functions, plot_filter_comparison,
)
import matplotlib.pyplot as plt
import numpy as np

## 1. Create the Gyroid Sphere Model

Same model as `hello_gyroid_sphere.py` but at a smaller resolution
for interactive exploration. Increase `voxel_size` denominator for higher res.

In [ ]:
RES = 128  # grid resolution — try 64, 128, 200
ENV.voxel_size = 12.0 / RES

sphere = Sphere(r=5)
gyroid = GyroidCube(12, center=True, lattice_param=1.5,
                    thresh1=-0.3, thresh2=0.3)
model = sphere & gyroid

model.render_volume()
print(f"Grid resolution: {model.grid.res_vector}")
print(f"Voxel count: {np.prod(model.grid.res_vector):,}")

## 2. One-Call Spectral Analysis

The `analyze_spectrum()` method computes the SDF, runs radial power spectrum
analysis, and returns bandwidth estimates plus safe stride values.

In [ ]:
result = model.analyze_spectrum(lowpass_cutoff=0.25, plot=False)

print(f"bandwidth_99:       {result['bandwidth_99']:.4f} cyc/vox")
print(f"bandwidth_95:       {result['bandwidth_95']:.4f} cyc/vox")
print(f"recommended_cutoff: {result['recommended_cutoff']:.4f} cyc/vox")
print(f"safe_stride:        {result['safe_stride']}")
print(f"stride-2 safe?      {'YES' if result['bandwidth_99'] < 0.25 else 'NO'}")

## 3. Radial Power Spectrum

The radial power spectrum averages FFT power in spherical shells at each
frequency radius. Geometry features appear as peaks at low frequencies;
staircase artifacts concentrate near Nyquist (0.5 cyc/vox).

In [ ]:
freq_bins = result['freq_bins']
power = result['power']
bw99 = result['bandwidth_99']

fig = plot_radial_spectrum(freq_bins, power,
                           cutoff=0.25, bandwidth=bw99,
                           title=f'Gyroid Sphere {RES}^3 — Radial Power Spectrum')
plt.show()

## 4. Cumulative Energy

What fraction of total spectral energy lies below each frequency?
The 99% line gives the geometry bandwidth. A sharp knee means clean
separation between geometry and staircase; a gradual rise means overlap.

In [ ]:
markers = {
    f'cutoff (0.25)': 0.25,
    f'bw99 ({bw99:.3f})': bw99,
}
fig = plot_cumulative_energy(freq_bins, power, markers=markers,
                             title=f'Gyroid Sphere {RES}^3 — Cumulative Energy')
plt.show()

## 5. 2D Spectrum Slices

Central slices through the 3D power spectrum at the DC plane.
Reveals whether staircase artifacts are isotropic or directional
(axis-aligned voxelization creates directional artifacts).

In [ ]:
sdf = result['sdf_raw']
fig = plot_spectrum_slices(sdf, title_prefix=f'Gyroid Sphere {RES}^3')
plt.show()

## 6. Before/After Butterworth Filter

Overlay the raw and filtered SDF spectra to see exactly what
the Butterworth removes. The gap between curves IS the staircase energy.

In [ ]:
sdf_filtered = apply_filter(sdf, 'butterworth2', cutoff=0.25)

fig = plot_filter_effect(sdf, sdf_filtered, cutoff=0.25,
                         title=f'Gyroid Sphere {RES}^3 — Filter Effect (Butterworth ord-2)')
plt.show()

## 7. Compare Individual Primitives

How do Sphere and GyroidCube differ spectrally? The Sphere should
have nearly all energy at very low frequencies, while the GyroidCube
shows harmonic peaks from the periodic wall structure.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sphere
r_sphere = sphere.analyze_spectrum(plot=False)
plot_radial_spectrum(r_sphere['freq_bins'], r_sphere['power'],
                     cutoff=0.25, bandwidth=r_sphere['bandwidth_99'],
                     ax=axes[0], title=f'Sphere {RES}^3')
axes[0].text(0.05, 0.05,
             f"bw99={r_sphere['bandwidth_99']:.4f}\n"
             f"stride={r_sphere['safe_stride']}",
             transform=axes[0].transAxes, fontsize=10,
             verticalalignment='bottom',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# GyroidCube
r_gyroid = gyroid.analyze_spectrum(plot=False)
plot_radial_spectrum(r_gyroid['freq_bins'], r_gyroid['power'],
                     cutoff=0.25, bandwidth=r_gyroid['bandwidth_99'],
                     ax=axes[1], title=f'GyroidCube {RES}^3')
axes[1].text(0.05, 0.05,
             f"bw99={r_gyroid['bandwidth_99']:.4f}\n"
             f"stride={r_gyroid['safe_stride']}",
             transform=axes[1].transAxes, fontsize=10,
             verticalalignment='bottom',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.tight_layout()
plt.show()

print(f"Sphere:     bw99={r_sphere['bandwidth_99']:.4f}, safe_stride={r_sphere['safe_stride']}")
print(f"GyroidCube: bw99={r_gyroid['bandwidth_99']:.4f}, safe_stride={r_gyroid['safe_stride']}")

## 8. Resolution Comparison

How does grid resolution affect the spectral content and stride safety?
At lower resolution, thin features occupy fewer voxels, pushing their
spectral content closer to the Butterworth cutoff.

In [ ]:
resolutions = [64, 128, 200]
results = []

for res in resolutions:
    ENV.voxel_size = 12.0 / res
    g = GyroidCube(12, center=True, lattice_param=1.5,
                   thresh1=-0.3, thresh2=0.3)
    g.render_volume()
    r = g.analyze_spectrum(plot=False)
    r['res'] = res
    results.append(r)
    del g

fig, ax = plt.subplots(figsize=(10, 6))
for r in results:
    mask = r['power'] > 0
    ax.semilogy(r['freq_bins'][mask], r['power'][mask],
                linewidth=1.5, label=f"{r['res']}^3 (bw99={r['bandwidth_99']:.3f})")

ax.axvline(0.25, color='r', linestyle='--', linewidth=1.5, label='cutoff (0.25)')
ax.axvline(0.5, color='0.5', linestyle=':', linewidth=1, label='Nyquist')
ax.set_xlabel('Frequency (cycles/voxel)')
ax.set_ylabel('Mean Power')
ax.set_title('GyroidCube — Resolution Comparison')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.55)
plt.show()

for r in results:
    print(f"  {r['res']}^3: bw99={r['bandwidth_99']:.4f}, "
          f"safe_stride={r['safe_stride']}, "
          f"stride-2 {'SAFE' if r['bandwidth_99'] < 0.25 else 'RISKY'}")

## 9. Filter Zoo — Transfer Function Profiles

Why Butterworth order-2? Here are seven filter options at the same cutoff.
Each trades off between passband preservation, transition sharpness,
stopband attenuation, and ringing artifacts.

- **Brick-wall**: Perfect separation but causes Gibbs ringing (oscillation near discontinuities)
- **Butterworth ord-2**: Gentle rolloff, no ringing, but attenuates geometry near cutoff
- **Butterworth ord-4/8**: Progressively sharper, approaching brick-wall behavior
- **Gaussian**: Maximally smooth (no ringing ever), but very gradual rolloff — destroys thin features before removing staircase
- **Hann/Tukey**: Windowed transitions — smooth taper avoids ringing while maintaining sharpness

In [ ]:
fig = plot_filter_transfer_functions(cutoff=0.25,
    title='Filter Transfer Functions (cutoff=0.25 cyc/vox)')
plt.show()

## 10. Filter Comparison on Gyroid Sphere SDF

Apply all seven filters to the same SDF and compare their effect on the
radial power spectrum. Top panel: filtered spectra overlaid. Bottom panel:
suppression ratio (how much each filter attenuates at each frequency).

In [ ]:
fig = plot_filter_comparison(sdf, cutoff=0.25,
    title=f'Gyroid Sphere {RES}^3 — Filter Comparison (cutoff=0.25)')
plt.show()

## 11. Real-Space Filter Artifacts — Central Z-Slice

Frequency-domain plots show *what* each filter removes; this shows *how it
looks* in physical space. The central Z-slice through the filtered SDF
reveals ringing (brick-wall), over-smoothing (Gaussian), and the
Butterworth sweet spot.

In [ ]:
filters_to_show = [
    ('brick',        'Brick-wall'),
    ('gaussian',     'Gaussian'),
    ('butterworth2', 'Butterworth ord-2'),
    ('butterworth4', 'Butterworth ord-4'),
    ('butterworth8', 'Butterworth ord-8'),
    ('hann',         'Hann window'),
    ('tukey',        'Tukey window'),
]
n_filters = len(filters_to_show)
mid_z = sdf.shape[2] // 2

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.ravel()

# Raw SDF slice
vmin, vmax = -3, 3
axes[0].imshow(sdf[:, :, mid_z].T, origin='lower', cmap='RdBu_r',
               vmin=vmin, vmax=vmax)
axes[0].set_title('Raw SDF', fontsize=10)
axes[0].set_ylabel('Y')

for i, (name, label) in enumerate(filters_to_show):
    filtered = apply_filter(sdf, name, cutoff=0.25)
    axes[i + 1].imshow(filtered[:, :, mid_z].T, origin='lower',
                       cmap='RdBu_r', vmin=vmin, vmax=vmax)
    axes[i + 1].set_title(label, fontsize=10)
    del filtered

for ax in axes:
    ax.set_xlabel('X')
    ax.tick_params(labelsize=7)

fig.suptitle(f'Gyroid Sphere {RES}^3 — Central Z-Slice (SDF contour=0 is the surface)',
             fontsize=12, y=1.02)
fig.tight_layout()
plt.show()